# 02 — Validation of GitSkills Content and Metadata

**Study:** Understanding the Structure and Reuse of AI Agent Skills in Open-Source Software Projects  
**Dataset:** GitSkills sample, July 2026

This notebook checks the data underlying the structural and cross-repository analyses. The checks cover deduplication, Git blob hashes, and supporting-file records. They are performed before interpreting identical hashes as exact-content overlap in RQ2 or using folder composition in RQ1.

An *artifact occurrence* is a recorded skill file in a repository. A *distinct content* is identified by its `file_sha`. The database marks one occurrence of each distinct content as its primary representative (`dedup_primary = 1`).


## 1. Database connection

The analysis reads `data/raw/agent_skills_sample.db` without modifying it. The database path is resolved from the project directory or its `notebooks/` subdirectory. No output from another notebook is required.


In [1]:
from pathlib import Path
import sqlite3
import hashlib
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DB_PATH = PROJECT_ROOT / "data" / "raw" / "agent_skills_sample.db"
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Expected GitSkills SQLite sample at: {DB_PATH}")

conn = sqlite3.connect(f"file:{DB_PATH.as_posix()}?mode=ro", uri=True)
print("Read-only database:", DB_PATH)
print("SQLite integrity check:", conn.execute("PRAGMA integrity_check").fetchone()[0])

Read-only database: C:\Users\mouzh\Documents\GitHub\MSR2027-GitSkills\data\raw\agent_skills_sample.db
SQLite integrity check: ok


## 2. Deduplication records

The dataset retains repeated occurrences of the same file content. We first examine how the `dedup_primary` and `content_fetched` fields are distributed. This distinguishes occurrences whose text is stored directly from those represented by another record with the same hash.


In [2]:
dedup_query = """
SELECT
    dedup_primary,
    content_fetched,
    COUNT(*) AS artifacts
FROM artifacts
GROUP BY dedup_primary, content_fetched
ORDER BY dedup_primary, content_fetched
"""

dedup_df = pd.read_sql_query(dedup_query, conn)

display(dedup_df)

,dedup_primary,content_fetched,artifacts
0,0,0,16770
1,0,1,16
2,1,1,13000


### 2.1 Primary representatives

Grouping by `file_sha` allows us to test whether each distinct content has exactly one primary record. A duplicated primary or a hash without a primary would undermine the distinct-content population used in RQ1.


In [3]:
primary_audit = pd.read_sql_query("""
SELECT file_sha,
       COUNT(*) AS occurrences,
       SUM(CASE WHEN dedup_primary = 1 THEN 1 ELSE 0 END) AS primaries
FROM artifacts
GROUP BY file_sha
""", conn)
print("Distinct hashes:", len(primary_audit))
print("Exactly one primary:", int((primary_audit.primaries == 1).sum()))
print("Zero primaries:", int((primary_audit.primaries == 0).sum()))
print("Multiple primaries:", int((primary_audit.primaries > 1).sum()))
assert len(primary_audit) == 13_000
assert primary_audit.primaries.eq(1).all(), "Unexpected primary representative counts"

Distinct hashes: 13000
Exactly one primary: 13000
Zero primaries: 0
Multiple primaries: 0


## 3. Availability and consistency of stored text

The following queries count fetched text records per hash and check whether hashes with multiple stored texts contain conflicting strings. Non-primary rows without fetched text are not necessarily missing skills: their content may be represented by the primary row.

String agreement is only an internal consistency check. The next section tests the stored text against its Git blob identifier.


In [4]:
hash_coverage_query = """
WITH hash_coverage AS (
    SELECT
        file_sha,
        COUNT(*) AS occurrences,
        SUM(
            CASE WHEN content IS NOT NULL THEN 1 ELSE 0 END
        ) AS available_contents
    FROM artifacts
    GROUP BY file_sha
)
SELECT
    COUNT(*) AS unique_hashes,
    SUM(CASE WHEN available_contents = 0 THEN 1 ELSE 0 END)
        AS hashes_without_content,
    SUM(CASE WHEN available_contents = 1 THEN 1 ELSE 0 END)
        AS hashes_with_one_content,
    SUM(CASE WHEN available_contents > 1 THEN 1 ELSE 0 END)
        AS hashes_with_multiple_contents
FROM hash_coverage
"""

hash_coverage_df = pd.read_sql_query(
    hash_coverage_query,
    conn
)

display(hash_coverage_df.T.rename(columns={0: "Value"}))

,Value
unique_hashes,13000
hashes_without_content,0
hashes_with_one_content,12984
hashes_with_multiple_contents,16


In [5]:
hash_validation_query = """
SELECT
    file_sha,
    COUNT(*) AS available_records,
    COUNT(DISTINCT content) AS distinct_contents
FROM artifacts
WHERE content IS NOT NULL
GROUP BY file_sha
HAVING COUNT(*) > 1
ORDER BY distinct_contents DESC, available_records DESC
"""

hash_validation_df = pd.read_sql_query(
    hash_validation_query,
    conn
)

print(
    "Hashes with multiple available content records:",
    len(hash_validation_df)
)

print(
    "Hashes associated with conflicting content:",
    (hash_validation_df["distinct_contents"] > 1).sum()
)

display(hash_validation_df.head(10))

Hashes with multiple available content records: 16
Hashes associated with conflicting content: 0


,file_sha,available_records,distinct_contents
0,000aacec4ac4c65f9e1722c7df7915cbec3c9112,2,1
1,00270e1db5b5373fc8e55219f6e9568febf1a68f,2,1
2,003f6b4f8818f7a45488ed24215c3357aac74c6d,2,1
3,0043c940d2437438935ae52def7383df2cd271d2,2,1
4,0050611abcfbe09ec5d368f0542a940c1968bc76,2,1
5,00532fe249835558951166b34875e53a753dafba,2,1
6,005696c89f667fcd18e7a98d69e35af252225188,2,1
7,007cda51c7ccaf84e3879f6a82917e90f17d4f3d,2,1
8,00a0867a14b520f86361fbac7319b2ad96db1ad7,2,1
9,00af7861bc7143c5d4586e033a3d028fc2f0f0f0,2,1


## 4. Verification of Git blob identifiers

A Git blob SHA-1 is calculated over a header containing the byte length of the file, a null byte, and the file's contents. We recompute that value from the stored UTF-8 text for every primary representative and compare it with `file_sha`.

This checks all 13,000 distinct contents in the sample. A matching hash supports exact byte-content identification; it says nothing about when a file was added to a repository or whether one repository copied it from another.


In [6]:
import hashlib

# Retrieve one content-bearing record per unique hash
full_validation_df = pd.read_sql_query(
    """
    SELECT file_sha, content
    FROM artifacts
    WHERE dedup_primary = 1
      AND content IS NOT NULL
    """,
    conn
)

def compute_git_blob_sha(content):
    data = content.encode("utf-8")
    header = f"blob {len(data)}\0".encode("utf-8")
    return hashlib.sha1(header + data).hexdigest()

# Independently compute Git blob hashes
full_validation_df["computed_sha"] = (
    full_validation_df["content"].apply(compute_git_blob_sha)
)

full_validation_df["matches"] = (
    full_validation_df["file_sha"]
    == full_validation_df["computed_sha"]
)

# Summarize results
total = len(full_validation_df)
matching = int(full_validation_df["matches"].sum())
mismatching = total - matching

print(f"Records tested: {total:,}")
print(f"Matching hashes: {matching:,}")
print(f"Non-matching hashes: {mismatching:,}")
print(f"Match rate: {matching / total * 100:.2f}%")

# Display any mismatches
display(
    full_validation_df.loc[
        ~full_validation_df["matches"],
        ["file_sha", "computed_sha"]
    ].head(10)
)

Records tested: 13,000
Matching hashes: 13,000
Non-matching hashes: 0
Match rate: 100.00%


,file_sha,computed_sha


### 4.1 Validation checks

The assertions below confirm that all expected primary texts were included and that their calculated hashes agree with the database. The expected count is fixed to this GitSkills sample, so the assertion would need updating for a different dataset release.


In [7]:
assert total == 13_000, f"Expected 13,000 primary texts, got {total}"
assert mismatching == 0, "Some primary content does not match its Git blob SHA-1"
assert matching == 13_000
assert int((hash_validation_df["distinct_contents"] > 1).sum()) == 0
print("PASS: All 13,000 primary contents match their Git blob SHA-1 hashes.")

PASS: All 13,000 primary contents match their Git blob SHA-1 hashes.


## 5. Supporting-file records

The dataset records nearby files and directories in `artifact_siblings`. The `sibling_count` field refers to files, whereas the table also includes directory entries. A direct comparison with every stored entry therefore overstates inconsistencies.

We inspect the composition collection flags and then compare `sibling_count` with the number of entries whose `entry_type` is `file`. These records describe the folder of the selected primary occurrence; another repository containing identical `SKILL.md` text may have different supporting files.


In [8]:
truncation_query = """
SELECT
    composition_truncated,
    COUNT(*) AS artifacts,
    MIN(sibling_count) AS min_siblings,
    MAX(sibling_count) AS max_siblings,
    AVG(sibling_count) AS mean_siblings
FROM artifacts
WHERE dedup_primary = 1
GROUP BY composition_truncated
ORDER BY composition_truncated
"""

truncation_df = pd.read_sql_query(
    truncation_query,
    conn
)

display(truncation_df)

,composition_truncated,artifacts,min_siblings,max_siblings,mean_siblings
0,NaN,10,NaN,NaN,NaN
1,0.0,11235,0.0,454.0,2.899243
2,1.0,1755,0.0,3218.0,8.458120


### 5.1 Comparing reported counts with stored entries

The query joins sibling entries to primary artifacts by repository and artifact path. It calculates both `stored_total` (files and directories) and `stored_files` (files only). The first difference is retained to show why entry types must be distinguished, not as the final validation result.


In [9]:
consistency_query = """
WITH stored_entries AS (
    SELECT
        repo_full_name,
        artifact_path,
        COUNT(*) AS stored_total,
        SUM(CASE WHEN entry_type = 'file' THEN 1 ELSE 0 END)
            AS stored_files,
        SUM(CASE WHEN entry_type = 'dir' THEN 1 ELSE 0 END)
            AS stored_dirs
    FROM artifact_siblings
    GROUP BY repo_full_name, artifact_path
)
SELECT
    a.repo_full_name,
    a.path,
    a.sibling_count,
    a.composition_truncated,
    COALESCE(s.stored_total, 0) AS stored_total,
    COALESCE(s.stored_files, 0) AS stored_files,
    COALESCE(s.stored_dirs, 0) AS stored_dirs
FROM artifacts a
LEFT JOIN stored_entries s
    ON a.repo_full_name = s.repo_full_name
   AND a.path = s.artifact_path
WHERE a.dedup_primary = 1
  AND a.composition_fetched = 1
"""

consistency_df = pd.read_sql_query(
    consistency_query,
    conn
)

consistency_df["difference"] = (
    consistency_df["sibling_count"]
    - consistency_df["stored_total"]
)

print("Records checked:", len(consistency_df))
print(
    "Exact matches:",
    (consistency_df["difference"] == 0).sum()
)
print(
    "Reported count exceeds stored entries:",
    (consistency_df["difference"] > 0).sum()
)
print(
    "Stored entries exceed reported count:",
    (consistency_df["difference"] < 0).sum()
)

display(
    consistency_df[
        consistency_df["difference"] != 0
    ].sort_values(
        "difference",
        ascending=False
    ).head(10)
)

Records checked: 12990
Exact matches: 9948
Reported count exceeds stored entries: 8
Stored entries exceed reported count: 3034


,repo_full_name,path,sibling_count,composition_truncated,stored_total,stored_files,stored_dirs,difference
10046,MichelKerkmeester/opencode--skilled-agent-loop...,.opencode/skills/system-spec-kit/SKILL.md,3218,1,500,468,32,2718
2387,huangwenxuangod/flash-recorder,SKILL.md,2542,1,500,448,52,2042
3113,samouraiworld/memba,SKILL.md,1760,1,500,436,64,1260
830,agisota/rox,skills/oh-my-claudecode/SKILL.md,1362,1,500,387,113,862
6601,thtang/alpha-persona-lab,yutinghao/SKILL.md,925,1,500,493,7,425
3192,mpsuesser/workspace,dotconfig/opencode/skills/__techs__/oxlint/SKI...,779,1,500,486,14,279
6308,asd1702/todongsan,SKILL.md,645,1,500,308,192,145
5876,controlplane-docs/docs,skill.md,562,1,500,433,67,62
12961,marian2js/opengoat,packages/core/src/core/skills/bundled/marketin...,2,0,3,2,1,-1
12971,Techfolk-AS/skill-bench,experiments/template/.claude/skills/javascript...,4,0,5,4,1,-1


In [10]:
consistency_summary_df = (
    consistency_df
    .assign(
        exact_match=lambda df: df["difference"] == 0,
        reported_exceeds=lambda df: df["difference"] > 0,
        stored_exceeds=lambda df: df["difference"] < 0
    )
    .groupby("composition_truncated", dropna=False)
    .agg(
        records=("difference", "size"),
        exact_matches=("exact_match", "sum"),
        reported_exceeds=("reported_exceeds", "sum"),
        stored_exceeds=("stored_exceeds", "sum"),
        max_difference=("difference", "max")
    )
    .reset_index()
)

display(consistency_summary_df)

,composition_truncated,records,exact_matches,reported_exceeds,stored_exceeds,max_difference
0,0,11235,8320,0,2915,0
1,1,1755,1628,8,119,2718


### 5.2 File-only comparison

Here the reported `sibling_count` is compared with `stored_files`. We also separate truncated and untruncated listings to determine whether discrepancies are associated with the collection limit.


In [11]:
# Compare reported sibling counts against stored file entries

consistency_df["file_difference"] = (
    consistency_df["sibling_count"]
    - consistency_df["stored_files"]
)

file_consistency_summary = (
    consistency_df
    .assign(
        exact_match=lambda df: df["file_difference"] == 0,
        reported_exceeds=lambda df: df["file_difference"] > 0,
        stored_exceeds=lambda df: df["file_difference"] < 0
    )
    .groupby("composition_truncated", dropna=False)
    .agg(
        records=("file_difference", "size"),
        exact_matches=("exact_match", "sum"),
        reported_exceeds=("reported_exceeds", "sum"),
        stored_exceeds=("stored_exceeds", "sum"),
        max_difference=("file_difference", "max")
    )
    .reset_index()
)

display(file_consistency_summary)

,composition_truncated,records,exact_matches,reported_exceeds,stored_exceeds,max_difference
0,0,11235,11235,0,0,0
1,1,1755,1745,10,0,2750


In [12]:
# Inspect records where sibling_count differs from stored file count

mismatched_files_df = consistency_df[
    consistency_df["file_difference"] != 0
].copy()

print("Total mismatches:", len(mismatched_files_df))

display(
    mismatched_files_df[
        [
            "repo_full_name",
            "path",
            "sibling_count",
            "composition_truncated",
            "stored_files",
            "stored_dirs",
            "file_difference"
        ]
    ]
    .sort_values("file_difference", ascending=False)
    .head(15)
)

Total mismatches: 10


,repo_full_name,path,sibling_count,composition_truncated,stored_files,stored_dirs,file_difference
10046,MichelKerkmeester/opencode--skilled-agent-loop...,.opencode/skills/system-spec-kit/SKILL.md,3218,1,468,32,2750
2387,huangwenxuangod/flash-recorder,SKILL.md,2542,1,448,52,2094
3113,samouraiworld/memba,SKILL.md,1760,1,436,64,1324
830,agisota/rox,skills/oh-my-claudecode/SKILL.md,1362,1,387,113,975
6601,thtang/alpha-persona-lab,yutinghao/SKILL.md,925,1,493,7,432
6308,asd1702/todongsan,SKILL.md,645,1,308,192,337
3192,mpsuesser/workspace,dotconfig/opencode/skills/__techs__/oxlint/SKI...,779,1,486,14,293
5876,controlplane-docs/docs,skill.md,562,1,433,67,129
6805,elixir-vibe/host_kit,SKILL.md,434,1,390,110,44
9180,ankushKun/pumpmyclaw,skill.md,447,1,442,58,5


### 5.3 Composition consistency checks

Composition metadata was fetched for 12,990 primary representatives. Of these, 11,235 are marked untruncated and 1,755 truncated. The file-only comparison identifies ten discrepancies, all in truncated listings. These assertions confirm that the notebook reproduces those sample-specific counts.

A truncation flag does not by itself mean that a file count is wrong. It indicates that the stored listing may be incomplete.


In [13]:
assert len(consistency_df) == 12_990
assert int((consistency_df["file_difference"] != 0).sum()) == 10
untruncated = consistency_df[consistency_df["composition_truncated"] == 0]
truncated = consistency_df[consistency_df["composition_truncated"] == 1]
assert len(untruncated) == 11_235
assert len(truncated) == 1_755
assert untruncated["file_difference"].eq(0).all()
assert int(truncated["file_difference"].eq(0).sum()) == 1_745
print("PASS: Composition consistency matches the original research results.")

PASS: Composition consistency matches the original research results.


## 6. Implications for the study

The validation found one primary representative for each of the 13,000 distinct hashes. All 13,000 primary texts reproduced their recorded Git blob SHA-1 values, and the available duplicate texts did not conflict. This supports the use of `file_sha` to group identical contents in the cross-repository analysis.

Supporting-file measurements require more care. Ten of the 12,990 primary records with fetched composition have file-count discrepancies, all among truncated listings. The structural analysis therefore describes the supporting files of representative occurrences, rather than assuming that every repository containing the same instructions has the same folder contents.

The checks establish consistency within the released sample, not the completeness of the original GitHub collection. Exact hash matches cannot establish copying direction, independent adoption, or semantic similarity. The database and analysis version should be retained together for reproduction.


In [14]:
conn.close()
print("Validation complete. Read-only SQLite connection closed.")

Validation complete. Read-only SQLite connection closed.
